# Trajectory Follow Comparison
Plots of the closed-loop simulations saved by `trajectory_generation.ipynb` (no simulation here).

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.sim.elbow_simulation import (simulation_path, load_simulations, load_reference,
                                  simulation_title, plot_reference, plot_angle, plot_torque, plot_activations)

## Import Data
The folder is built with `simulation_path`, the same function used to save, so the names always match (format `:g`: `0.40` → `0.4`, `0.0` → `0`).

In [ ]:
KIND = "minjerk_500ms" #"step"  #"minjerk_500ms"

W_REF = 0.5
INIT_ANGLE = 20   # [°]
ANGLE_REF = 90    # [°]

K_P = 1.0
K_I = 0.0
K_D = 0.0

DELAY_MS = 0.0# 150.0  # [ms], 0 = no delay
FEEDFORWARD = False

FOLDER_NAME = simulation_path(INIT_ANGLE, ANGLE_REF, W_REF, K_P, K_I, K_D, kind=KIND,
                              delay_ms=DELAY_MS, ff=FEEDFORWARD)

MODELS_STRUCTURE = []#["3L_64U"]
ACTIVATION_MODE = ["relu"]
OUTPUT_ACTIVATION_MODE = ["linear"]#, "linear_cos_decay"]

MODELS = ["LUT", "solver"]
for struct in MODELS_STRUCTURE:
    for act in ACTIVATION_MODE:
        for out in OUTPUT_ACTIVATION_MODE:
            MODELS.append(f"W_all_{struct}_{act}_{out}")

SAVE_PATH_PLOTS = FOLDER_NAME.replace("simulations", "plots", 1)
sims, parameters = load_simulations(FOLDER_NAME, labels=MODELS)
angle_ref = load_reference(FOLDER_NAME)

print("Loaded:", list(sims))
pd.Series({k: v for k, v in parameters.items() if k != "allocators"})

## Summary

In [ ]:
rows = {}
for name, df in sims.items():
    error = df.theta - df.theta_d
    rows[name] = {"final angle [deg]": df.theta.iloc[-1],
                  "final error [deg]": error.iloc[-1],
                  "max angle [deg]": df.theta.max(),
                  "RMS error [deg]": (error ** 2).mean() ** 0.5,
                  "final tau_req [Nm]": df.tau_req.iloc[-1]}

summary = pd.DataFrame(rows).T
summary.round(3)

## Plots

In [ ]:
PLOT_T_MIN = None # 0.500 # [s], None = from the start
PLOT_T_MAX = None # 1.500 # [s], None = to the end

title = simulation_title(INIT_ANGLE, ANGLE_REF, W_REF, K_P, K_I, K_D, kind=KIND,
                         delay_ms=DELAY_MS, ff=FEEDFORWARD)
window = dict(t_min=PLOT_T_MIN, t_max=PLOT_T_MAX)

#plot_reference(angle_ref, title=f"Desired angle ({INIT_ANGLE}° → {ANGLE_REF}°)", save_path=SAVE_PATH_PLOTS, **window)
plot_angle(sims, title=f"Elbow angle ({title})", save_path=SAVE_PATH_PLOTS, **window)
plot_torque(sims, title=f"Torque ({title})", save_path=SAVE_PATH_PLOTS, **window)
plot_activations(sims, title=f"Activations ({title})", save_path=SAVE_PATH_PLOTS, **window)